In [1]:
#| default_exp stack

In [2]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
import sys, traceback
from tempfile import TemporaryDirectory

Which of your own packages is installed, where an import of it resolves to, and which one raised.

Nothing here imports the packages it reports on. Every answer comes from import metadata, from a path, or from the text of a traceback. A package that cannot be imported is still described.

In [3]:
#| export
from __future__ import annotations

In [4]:
#| export
import re

In [5]:
#| export
from pathlib import Path

In [6]:
#| export
from fastcore.basics import first, ifnone

In [7]:
#| export

In [8]:
#| export
#: The packages a blame report looks for by default. A caller with its own family passes `family=`.
FAMILY = ['gheasy', 'dockeasy', 'vpseasy', 'cfeasy', 'dhrishti', 'nbdev', 'fastship', 'pullup']

`FAMILY` is the default list: the packages this project is built out of, and the only ones a report attributes anything to. A frame in `json` or `asyncio` is somebody else's code and is passed over. Every function taking `family=` uses that list instead, and an empty `family` falls back to `FAMILY`.

In [9]:
#| export
_FRAME = re.compile(r'File "([^"]+)", line (\d+), in (\S+)')

In [10]:
#| export
def installed(name):
    "Whether `name` is importable, without importing it."
    from importlib.util import find_spec
    try: return find_spec(name) is not None
    except (ImportError, ValueError): return False

`installed` answers whether an import would find `name`, not whether it would succeed. A module found by its spec can still fail to import: `vpseasy` ships an f-string that only parses on Python 3.12, so `installed('vpseasy')` is `True` under 3.11 and importing it raises `SyntaxError`. A caller that needs the module has to import it and handle the failure.

A name that is not a usable module name answers `False` rather than raising.

In [11]:
installed('nbdev'), installed('no_such_package_here'), installed('')

(True, False, False)

In [12]:
#| export
def _origin(name):
    "Where importing `name` would actually read from, without importing it."
    from importlib.util import find_spec
    try: spec = find_spec(name)
    except (ImportError, ValueError, ModuleNotFoundError): return None
    if spec is None: return None
    where = first(spec.submodule_search_locations or ()) or spec.origin
    return Path(where) if where else None

In [13]:
#| export
def _version(name):
    from importlib import metadata
    try: return metadata.version(name)
    except Exception: return ''

In [14]:
#| export
def _vendored(path):
    "Whether the path is inside an installed environment rather than a working tree."
    parts = set(Path(path).parts)
    return bool(parts & {'site-packages', 'dist-packages', '.venv', 'venv'})

In [15]:
#| export
def _roots(checkouts): return [Path(c).expanduser().resolve() for c in checkouts]

In [16]:
#| export
def checkout_for(name, checkouts):
    "The open checkout that *is* this package, matched by `<root>/<name>/__init__.py`, not repo name."
    for root in _roots(checkouts):
        for inner in (root/name, root/'src'/name):
            if (inner/'__init__.py').exists(): return str(root)
    return ''

In [17]:
#| export
def _row(name, roots):
    "One package: where an import resolves to, and whether that copy is the open checkout."
    path = _origin(name)
    local = checkout_for(name, roots)
    from_source = bool(path) and not _vendored(path)
    return {'name': name, 'installed': path is not None, 'version': _version(name),
            'path': str(path or ''), 'checkout': local, 'editable': from_source,
            'source': ('checkout' if from_source else 'site-packages') if path else '',
            'shadowed': bool(local) and bool(path) and not from_source}

In [18]:
#| export
def survey(family=(), checkouts=()):
    "One row each: installed, version, the resolved copy, and whether a checkout is shadowed."
    roots = _roots(checkouts)
    return [_row(name, roots) for name in (list(family) or FAMILY)]

`survey` describes one package per row: whether an import finds it, the version its metadata reports, the path the import resolves to, and the checkout you have open for it.

`editable` and `source` say the same thing two ways. The resolved copy is a working tree unless its path lies under `site-packages`, `dist-packages`, `.venv` or `venv`. `shadowed` means you have the checkout open and imports are still reading an installed copy, so nothing you edit runs.

A package that is not installed has an empty `version`, `path` and `source`, and `installed` is `False`.

In [19]:
[{k: r[k] for k in ('name', 'installed', 'source', 'shadowed')}
 for r in survey(family=['pullup', 'nbdev', 'dhrishti'])]

[{'name': 'pullup',
  'installed': True,
  'source': 'checkout',
  'shadowed': False},
 {'name': 'nbdev',
  'installed': True,
  'source': 'site-packages',
  'shadowed': False},
 {'name': 'dhrishti', 'installed': False, 'source': '', 'shadowed': False}]

The checkout is matched by what is inside it rather than by its folder name, so a fork called `nbdev-fork` still holds the `nbdev` package and is still the checkout for it.

In [20]:
#| hide
#| exec_doc
tmp = TemporaryDirectory(); root = Path(tmp.name)

In [21]:
fork = root/'nbdev-fork'; (fork/'nbdev').mkdir(parents=True)
(fork/'nbdev'/'__init__.py').write_text('')
row = survey(family=['nbdev'], checkouts=[fork])[0]
{k: row[k] for k in ('source', 'editable', 'shadowed')}

{'source': 'site-packages', 'editable': False, 'shadowed': True}

In [22]:
#| hide
test_eq(row['checkout'], str(fork))
test_eq([r['name'] for r in survey()], FAMILY)
for r in survey(family=['pullup', 'nbdev', 'dhrishti']):
    test_eq(bool(r['source']), r['installed'])
    test_eq(r['editable'], r['source'] == 'checkout')

In [23]:
#| export
def frames(text):
    "Every `File ..., line N, in f` in a traceback, oldest first, as the terminal printed it."
    return [{'file': m.group(1), 'line': int(m.group(2)), 'fn': m.group(3)}
            for m in _FRAME.finditer(str(text or ''))]

`frames` reads the `File ..., line N, in f` lines out of whatever text it is given, in the order the traceback printed them: the outermost caller first, the frame that raised last. It takes text rather than a traceback object, because what a panel has is captured output. Text with no frames in it, and `None`, both give an empty list.

The rest of this page works on a real traceback. Here is a package that raises.

In [24]:
#| exec_doc
pkg = root/'blowup'; pkg.mkdir()
(pkg/'__init__.py').write_text('')
(pkg/'core.py').write_text("""def push(remote):
    raise RuntimeError(f"no upstream for {remote}")

def sync():
    push("origin")
""")
sys.path.insert(0, str(root))

In [25]:
from blowup.core import sync
try: sync()
except RuntimeError: tb = traceback.format_exc()
print(tb.replace(str(root), '/proj'))   # the temp root stands in for a checkout

Traceback (most recent call last):
  File "<ipython-input-25-8bf6300fe37f>", line 2, in <module>
    try: sync()
         ^^^^^^
  File "/proj/blowup/core.py", line 5, in sync
    push("origin")
  File "/proj/blowup/core.py", line 2, in push
    raise RuntimeError(f"no upstream for {remote}")
RuntimeError: no upstream for origin



The deepest frame is the one that ran the `raise`, and it is the last one `frames` returns.

In [26]:
{**frames(tb)[-1], 'file': frames(tb)[-1]['file'].replace(str(root), '/proj')}

{'file': '/proj/blowup/core.py', 'line': 2, 'fn': 'push'}

In [27]:
#| hide
test_eq([f['fn'] for f in frames(tb)][-2:], ['sync', 'push'])
test_eq(frames(tb)[-1]['line'], 2)
test_eq(frames(''), []); test_eq(frames(None), [])
test_eq(frames('nothing here'), [])

In [28]:
#| export
def _package_of(path, names):
    "The family package a file belongs to."
    parts = Path(path).parts
    for marker in ('site-packages', 'dist-packages'):
        if marker in parts:
            i = parts.index(marker) + 1
            found = parts[i] if i < len(parts) else ''
            return found if found in names else ''
    for part in reversed(parts):
        if part in names: return part
    return ''

`_package_of` names the family package a file belongs to, by two rules. Inside an installed environment only the directory immediately below `site-packages` or `dist-packages` counts, so a third-party package that vendors a copy of one of yours is not blamed for it. Anywhere else the deepest matching path part wins, which is how a checkout is laid out: `~/code/gheasy/gheasy/repo.py`.

A file belonging to no family package gives `''`.

In [29]:
(_package_of('/home/me/code/gheasy/gheasy/repo.py', ['gheasy']),
 _package_of('/x/.venv/lib/python3.11/site-packages/vendorpkg/gheasy/repo.py', ['gheasy']),
 _package_of('/usr/lib/python3.11/json/decoder.py', ['gheasy']))

('gheasy', '', '')

In [30]:
#| hide
test_eq(_package_of('/x/.venv/lib/python3.11/site-packages/gheasy/repo.py', FAMILY), 'gheasy')
test_eq(_package_of('<stdin>', FAMILY), '')

In [31]:
#| export
def _is_error(line):
    "A line that reads like `SomeError: message` or a bare `SomeError`, not a frame, a source echo or the header."
    if not line or line.startswith((' ', 'File "', '^', '~', '|', 'Traceback (')): return False
    return ':' in line or line.rstrip().replace('.', '').isidentifier()

The error line is the last line of a traceback that names an exception, and `_is_error` has to tell it from the other three kinds of line a traceback holds: the header, a `File ...` frame, and the indented echo of the source that raised. A source echo often ends in a colon, and an exception raised with no message has no colon at all, so neither the colon nor its absence settles it. Indentation and the header text do the rest.

In [32]:
[_is_error(l) for l in ('Traceback (most recent call last):',
                        '  File "core.py", line 2, in push',
                        '    if not upstream:',
                        'RuntimeError',
                        'RuntimeError: no upstream for origin')]

[False, False, False, True, True]

In [33]:
#| hide
test_eq([_is_error(l) for l in ('Traceback (most recent call last):', '  File "core.py", line 2, in push',
                                '    if not upstream:', 'RuntimeError', 'RuntimeError: no upstream')],
        [False, False, False, True, True])

In [34]:
#| export
def blame(text, checkouts=(), family=None):
    "Which package raised (the deepest family frame) and where to open it, preferring a checkout."
    text = str(text or '')
    names = list(family or ()) or FAMILY
    roots = _roots(checkouts)
    family = [f | {'package': pkg} for f in frames(text)
              if (pkg := _package_of(f['file'], names))]
    found = family[-1] if family else None
    error = ifnone(first(reversed(text.strip().splitlines()), _is_error), '').rstrip()
    if found is None:
        return {'package': '', 'file': '', 'line': 0, 'fn': '', 'error': error, 'open': ''}
    open_at = found['file']
    checkout = checkout_for(found['package'], roots)
    if checkout and not str(open_at).startswith(str(checkout)):
        parts = Path(found['file']).parts
        i = parts.index(found['package'])
        candidate = Path(checkout).joinpath(*parts[i:])
        if candidate.exists(): open_at = str(candidate)
    return {'package': found['package'], 'file': found['file'], 'line': found['line'],
            'fn': found['fn'], 'error': error, 'open': open_at, 'checkout': checkout or '',
            'version': _version(found['package'])}

`blame` answers one question: which of your packages raised, and where to open it.

The deepest family frame is the one reported. The frames above it are whoever called, and a package is not at fault for calling something that failed. `error` is the exception line as the traceback printed it. `open` is the file to point an editor at. It is `file` unless you have a checkout of that package and the same file exists inside it, in which case editing `file` would mean editing a copy that a reinstall discards.

A traceback with none of your packages in it still reports `error`, with `package`, `file`, `fn` and `open` empty and `line` zero. Empty text and `None` give the same blanks. `blame` never raises.

In [35]:
r = blame(tb, checkouts=[root], family=['blowup'])
{k: (v.replace(str(root), '/proj') if isinstance(v, str) else v) for k, v in r.items()}

{'package': 'blowup',
 'file': '/proj/blowup/core.py',
 'line': 2,
 'fn': 'push',
 'error': 'RuntimeError: no upstream for origin',
 'open': '/proj/blowup/core.py',
 'checkout': '/proj',
 'version': ''}

In [36]:
#| hide
r = blame(tb, checkouts=[root], family=['blowup'])
test_eq((r['package'], r['fn'], r['line']), ('blowup', 'push', 2))
test_eq(r['error'], 'RuntimeError: no upstream for origin')
test_eq(Path(r['open']).read_text().splitlines()[r['line']-1].strip()[:5], 'raise')
test_eq(blame(tb, family=['nbdev'])['package'], '')
test_eq(blame(tb, family=['nbdev'])['error'], r['error'])
test_eq(blame(None), blame(''))

The traceback names the copy that ran. `open` names the copy you can edit, which is a different file when the code that raised was installed rather than checked out.

In [37]:
other = root/'blowup-fork'; (other/'blowup').mkdir(parents=True)
(other/'blowup'/'__init__.py').write_text(''); (other/'blowup'/'core.py').write_text('')
moved = blame(tb, checkouts=[other], family=['blowup'])
moved['file'] == moved['open'], moved['open'].startswith(str(other))

(False, True)

An exception raised with no message still has an error line, and it is the class name alone.

In a chained traceback, `error` is the last exception raised and the frame is the deepest family frame anywhere in the text. Those two are not always parts of the same exception.

In [38]:
#| hide
def bare(): raise RuntimeError
try: bare()
except RuntimeError: test_eq(blame(traceback.format_exc())['error'], 'RuntimeError')

def widen(): raise ValueError('cannot size the page')
try:
    try: sync()
    except RuntimeError: widen()
except ValueError: chained = traceback.format_exc()
test_eq(blame(chained, family=['blowup'])['error'], 'ValueError: cannot size the page')
test_eq(blame(chained, family=['blowup'])['fn'], 'push')

In [39]:
#| hide
sys.path.remove(str(root)); tmp.cleanup()